In [ ]:
#Cell 1: Setup and Directory Creation
import os
baseDir = '/Users/shrey/NeuroSpeech/speechBCI-main'
os.makedirs(baseDir+'/derived/rnns', exist_ok=True)

In [ ]:
# Cell 2: Training Command for HM-RNN
# First, you need to create the model file: neuralDecoder/models/hierarchical_multiscale_gru.py
# See the separate artifact for the complete implementation

# Training command for HM-RNN
get_ipython().run_cell_magic('bash', '', '''
python3 -m neuralDecoder.main \\
    dataset=speech_release_baseline \\
    model=hierarchical_multiscale_gru \\
    learnRateDecaySteps=10000 \\
    nBatchesToTrain=10000 \\
    learnRateStart=0.02 \\
    model.nLayers=3 \\
    model.nUnitsPerLayer=[512,768,1024] \\
    model.downsampleFactors=[1,4,16] \\
    model.downsampleMethod=max_pool \\
    model.upsampleMethod=repeat \\
    model.combineMethod=concat \\
    model.stack_kwargs.kernel_size=32 \\
    model.dropout=0.3 \\
    outputDir=outputDir = '/Users/shrey/NeuroSpeech/speechBCI-main/outputs'
''')

In [ ]:
# Cell 3: Load Output Snapshot
# Visualize outputs - now with multi-scale analysis
import scipy.io
import matplotlib.pyplot as plt
import numpy as np

dat = scipy.io.loadmat(baseDir+'/derived/rnns/hmrnnRelease/outputSnapshot')
print(dat.keys())

In [ ]:
# Cell 4: Visualize Final Logits
# Visualize final logits
plt.figure(figsize=(12, 4))
plt.imshow(dat['logitsSnapshot'].T, aspect='auto')
plt.title('Final Logits Output')
plt.xlabel('Time')
plt.ylabel('Character Classes')
plt.colorbar()
plt.show()

In [ ]:
# Cell 5: Visualize Input Features
# Visualize input features
plt.figure(figsize=(12, 4))
plt.imshow(dat['inputFeaturesSnapshot'].T, aspect='auto')
plt.title('Input Neural Features')
plt.xlabel('Time')
plt.ylabel('Feature Dimensions')
plt.colorbar()
plt.show()

In [ ]:
# Cell 6: Visualize Multi-Scale Layer Outputs
# If you've saved intermediate layer outputs, visualize them
if 'layer0_output' in dat:
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))

    # Fast layer (every timestep)
    axes[0].imshow(dat['layer0_output'].T, aspect='auto')
    axes[0].set_title('Layer 0: Fast (downsample=1) - Phoneme-level')
    axes[0].set_ylabel('Hidden Units')

    # Medium layer (every 4 timesteps)
    axes[1].imshow(dat['layer1_output'].T, aspect='auto')
    axes[1].set_title('Layer 1: Medium (downsample=4) - Syllable-level')
    axes[1].set_ylabel('Hidden Units')

    # Slow layer (every 16 timesteps)
    axes[2].imshow(dat['layer2_output'].T, aspect='auto')
    axes[2].set_title('Layer 2: Slow (downsample=16) - Word-level')
    axes[2].set_xlabel('Time')
    axes[2].set_ylabel('Hidden Units')

    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 7: Print Output Shapes
print(f"Input features shape: {dat['inputFeaturesSnapshot'].shape}")
print(f"Logits shape: {dat['logitsSnapshot'].shape}")